In [2]:
# # _회귀분석 이런 저런 방법들. 
# 일단 SLM 회귀분석 방법 및 추정량의 불편성, 일치성, 모의실험으로 보이기.

import os
import numpy as np
import matplotlib.pyplot as plt


In [10]:
def save_chart(fname, outdir, fig=None, dpi=300, bbox_inches='tight', transparent=False):
    if fig is None:
        fig = plt.gcf()
    os.makedirs(outdir, exist_ok=True)
    name, ext = os.path.splitext(fname)  # fname(파일)을 이름과 익스텐션으로 분리.
    ext = ext or '.png'
    path = os.path.join(outdir, name + ext)  # outdir에 fname을 짤라서 그림 이름으로 지정하네? 
    fig.savefig(path, dpi=dpi, bbox_inches=bbox_inches, transparent=transparent)
    return path

# numpy as np 필수 
# 상관관계에 딱 맞는 계열 쌍 생성 

def rngd_xy_rho(rho, n_pop, seed) :
    rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
                                       # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 

    # 3.1. 난수 생성: z1, z2  
    z1 = rng.normal(size=n_pop)
    z2 = rng.normal(size=n_pop)

    # 3.2. 표준화: 결국 표준정규변수 z1, z2, 서로 독립인 상태. 상관계수 0. 
    z1 = (z1 - z1.mean()) / z1.std(ddof=1)
    z2 = (z2 - z2.mean()) / z2.std(ddof=1)

    # 3.3. 직교화 (Gram-Schmidt) 
    z2 = z2 - np.dot(z1, z2) / np.dot(z1, z1) * z1
    z2 = z2 / z2.std(ddof=1)

    # 3.4. 원하는 상관계수 생성, x, y는 정확히 모 상관계수로 생성.
    x = z1
    y = rho * z1 + np.sqrt(1 - rho**2) * z2

    # 3.5. 다시 표준화, x, y는 정확히 상관계수에서 생성된 변수. 표준화된 변수. 
    #   이것이 모의실험에 투입되는 변수임. 
    #   모상관계수가 그렇다고 해도 표본 상관계수는 차이날 수 있음. 이것을 극복하는 것.
    #   교과서에 보여줄 수 있는 그림. 그것임. 
    x = (x - x.mean()) / x.std(ddof=1)
    y = (y - y.mean()) / y.std(ddof=1)

    r = np.corrcoef(x, y)[0, 1]  # 실험 상관계수, 이론 값과 일치하도록.
    return x, y 


In [3]:
# 단순 회귀 실험용 자료 생성, 모의실험 
# 엄밀하게는 주어진 X, 이것은 확률변수 아님. 고로 평균함수도 마찬가지.
# 종속변수는 평균함수에 임의 요소 추가. 
seed = 135678942 
rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
                                       # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 
def rngd_slm_y(beta_0, beta_1, x_dat, slm_std) :
    n_size = len(x_dat)       # x는 non-random.
    # 난수 생성: 
    slm_e = rng.normal(loc = 0, scale= slm_std, size=n_size)
    y = beta_0 + beta_1 * x_dat + slm_e 
    return y 



In [4]:
# def main()   # 필요시 여기부터 지정하자. 
# 일단 파일 읽어서 X 고정하는 과정. 
# # 로컬 파이썬에서는 문제 없음. (이것은 로컬 파이썬 단독 실행시)
import os 
import pandas as pd

input_dir = 'in_files' 
output_dir = 'o_files'
output_fig_dir = 'figures'

input_file = 'cs_nns_gndr_hgt.xlsx'
input_sheet = 'data'
output_file = 'test.csv' or 'test.xlsx'

# 파일 경로 결합
in_file_path = os.path.join(input_dir, input_file)
o_file_path = os.path.join(output_dir, output_file)

# 데이터 파일 읽기. 
df_dat = pd.read_excel(in_file_path, sheet_name=input_sheet)

# 생성 파일 z_table(그때 그때) 저장
# z_table.to_csv('o_files/z_table.csv')  ==> (o_file_path) 이렇게 교체해야 할 듯. 
# z_table.to_excel('o_files/z_table.xlsx')

df_dat.info()
# # # 변수 i, gender, ht. 관측치 20128개. 
n_ttl = len(df_dat)
print(n_ttl)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20128 entries, 0 to 20127
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   i       20128 non-null  int64  
 1   gender  20128 non-null  int64  
 2   ht      20128 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 471.9 KB
20128


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
# scikit-learn 모듈. 머신러닝 모듈임.
from sklearn.linear_model import LinearRegression

n_iter = 450
n_size = 210 
x_dat = df_dat['ht'].head(n_size)
beta_0 = 30 
beta_1 = 0.5
slm_std = 15 
seed = 135678942


# X = x_dat.to_numpy()  # 2차원 배열(행렬: 샘플수, 특성수), 그런데 시리즈는 1차원 배열임.
Xv = x_dat.to_numpy().reshape(-1,1) 
            # 대안1. numpy 후 reshape 적용. 1차원 배열 (N,) --> 2차원 배열 (N, 1)로 변환
            # 대안2: 시리즈에서 읽을 때 [] 두번 감싸고 넘파이. x_dat = df_dat[['ht]].to_numpy()  
X = np.c_[np.ones(( len(Xv), 1)) , Xv]

rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
                                       # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 
                                       # 아래 함수 내에 난수 생성 들어감.

# 여기서 부터 반복 예정, 실험이니까, 다음 셀. 
y = rngd_slm_y(beta_0, beta_1, x_dat, slm_std) 
y = y.to_numpy()          


beta = np.linalg.inv(X.T @ X) @ (X.T @ y )
beta = beta.reshape(-1, 1)  # -1은 알아서 맞추라는 뜻, 1은 열의 갯수 1. re-assign 해야.
                     # (1, -1) 행 1개, 나머지는 알아서.
                     # (-1, 2) 열 2개, 나머지는 알아서. 
# print(beta)
# print(beta.shape)

# model = LinearRegression()  
# model.fit(X,y)
# beta_1_hat = model.coef_       # 이건 array
# beta_0_hat = model.intercept_  # 이건 float 
#     # print(model.coef_)       # 기울기
#     # print(model.intercept_)  # 절편, 모형에 상수항 자동 추가네.
#     # print(beta_0_hat, beta_1_hat) 
#     # type(beta_0_hat)  # 이건  numpy.float 
#     # type(beta_1_hat)  # 이건 numpy.ndarray
# beta = ((model.intercept_, model.coef_[0]))  # 리스트로 묶어. 
# beta_hat = np.array(beta)  # ndarray로 변환.

# print("단 1회 실행 결과 ")
# print(f"표본크기 {n_size}, 절편 {beta_0}, 기울기 {beta_1} ")
# print(f" 추정치 절편 {beta_0_hat}, 기울기 {beta_1_hat} ")
# print(beta)      # 이건 1행으로 짝 맞춰 계속 나열, 즉 리스트 형태. 1회 실행이라 1행으로 2개 나열/리스트
# print(beta_hat)  # 이건 행마다 짝을 맞추고, 행을 아래로 나열. 즉 행렬 형태. 1회 실행이라 1행, 2열. 




[[-2.86597201]
 [ 0.69304299]]
(2, 1)


In [57]:
# 이 셀은 위 셀의 OLS를 여러번 반복한 결과 정리. 

import os
import numpy as np
import matplotlib.pyplot as plt
# scikit-learn 모듈. 머신러닝 모듈임.
from sklearn.linear_model import LinearRegression

n_iter = 7
n_size = 4321 
x_dat = df_dat['ht'].head(n_size)
beta_0 = 30 
beta_1 = 0.5
slm_std = 15 
seed = 135678942


# X = x_dat.to_numpy()  # 2차원 배열(행렬: 샘플수, 특성수), 그런데 시리즈는 1차원 배열임.
X = x_dat.to_numpy().reshape(-1,1) 
            # 대안1. numpy 후 reshape 적용. 1차원 배열 (N,) --> 2차원 배열 (N, 1)로 변환
            # 대안2: 시리즈에서 읽을 때 [] 두번 감싸고 넘파이. x_dat = df_dat[['ht]].to_numpy()  

rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
                                       # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 
                                       # 아래 함수 내에 난수 생성 들어감.

beta = []   # 반복 결과를 list로 받자. 나중에 ndarray로 전환.
for i in range( n_iter) :  # range(0, n_iter, 1) 과 동일함....
# 여기서 부터 반복 예정, 실험이니까. 
    y = rngd_slm_y(beta_0, beta_1, x_dat, slm_std) 
    y = y.to_numpy()          
    model = LinearRegression()  
    model.fit(X,y)
    beta_1_hat = model.coef_       # 이건 array
    beta_0_hat = model.intercept_  # 이건 float 
    print("몇 번째 반복인가? ", i, ", 기울기:",  beta_1_hat)  # 반복하면서 추정한다는 걸 보여주자. 

    beta.append((model.intercept_, model.coef_[0]))  # 리스트로 묶어서 추가한다고. 
    # 최종 결과, beta. 리스트임. 

beta_hat = np.array(beta)  # ndarray로 변환.
# print(beta)      # 이건 1행으로 짝 맞춰 계속 나열, 즉 리스트 형태.
# print(beta_hat)  # 이건 행마다 짝을 맞추고, 행을 아래로 나열. 즉 행렬 형태.  

mean_b_0 = np.mean( beta_hat[:,0] ) # 인덱스 0인 array 평균. 
mean_b_1 = np.mean( beta_hat[:,1] ) # 인덱스 1인 array 평균. 
se_b_1 = np.std(beta_hat[:,1], ddof=1)  
print(f"반복횟수 {n_iter}, 표본크기 {n_size}, 절편 {beta_0}, 기울기 {beta_1} ")
print(f"반복 추정치  절편의 평균, 이 값이 참 절편(모수){beta_0}에 가까운가?", mean_b_0)
print(f"반복 추정치 기울기의 평균, 이 값이 참 기울기(모수){beta_1}에 가까운가?", mean_b_1)
print(f"표본 크기 ({n_size})가 증가할 때, 반복 추정 기울기의 표준오차({se_b_1})는 0에 접근하나?") 




몇 번째 반복인가?  0 , 기울기: [0.49672928]
몇 번째 반복인가?  1 , 기울기: [0.48603931]
몇 번째 반복인가?  2 , 기울기: [0.48476805]
몇 번째 반복인가?  3 , 기울기: [0.47642553]
몇 번째 반복인가?  4 , 기울기: [0.4770952]
몇 번째 반복인가?  5 , 기울기: [0.51389282]
몇 번째 반복인가?  6 , 기울기: [0.49479342]
반복횟수 7, 표본크기 4321, 절편 30, 기울기 0.5 
반복 추정치  절편의 평균, 이 값이 참 절편(모수)30에 가까운가? 31.706699716305334
반복 추정치 기울기의 평균, 이 값이 참 기울기(모수)0.5에 가까운가? 0.48996337484422614
표본 크기 (4321)가 증가할 때, 반복 추정 기울기의 표준오차(0.013121090794170514)는 0에 접근하나?


In [15]:

# 7. 메인() 일괄 실행. 이런 식으로 하려면, 위 단추 Run All 클릭.
# 이미 위 셀들이 실행된 적이 있으면, 이 셀만 실행하면 OK
# 그렇지 않으면, 모든 셀 일괄 실행. 즉 Run All

if __name__ == "__main__":
    # 실행 전 설정
    # 미리 준비하면서 정보를 파악해 두어야 함.
    # 아웃풋 폴더
    # 데이터 폴더, 데이터 파일 이름, 데이터 시트 이름,
    # 데이터의 그룹, 층, 범주 이름 등
    # 난수생성관련 조건 등 
    output_fig_dir = 'figures'
    output_fig_file = 'sta_09001_corr_01.png'
    output_dir = 'o_files'
    input_dir = 'in_files' 
    input_file = 'cs_nns_hgt_wgt.xlsx' # 변수명: x, y 
    input_sheet = 'hwght'
    strat = 'gender'  # stratum variable, check with data file, 2그룹인 경우.
    rrank = 'rnrk' # 이런 것은 미리 알고 있어야... 데이터 파악 단계에서. 
#    n_pop = 100    # 연습용 모집단 크기, 모의실험 대상.
    seed = 12348215 
    n_pop = 100  # 모집단 크기인 셈. 샘플 사이즈, 생성할 관측 갯수.
    sub_c = 3
    sub_r = 3
    nfig = sub_r*sub_c 

